# MailSense — Controlled Kaggle Training

This notebook runs the finalized MailSense pipeline for the three locked models:

1. TF-IDF + Logistic Regression
2. LSTM + pretrained Word2Vec embeddings
3. Pretrained BERT fine-tuning

All models use the same processed dataset and the same held-out test split. The notebook only orchestrates the finalized scripts; model logic remains in `src/`.

## Kaggle input

Create a Kaggle Dataset containing the project files. The notebook expects:

```text
/kaggle/input/mailsense/
├── src/
├── configs/
├── data/
│   └── Ask0729-fixed.txt
└── embeddings/
    └── word2vec.bin
```

For BERT, Kaggle Internet access must be enabled unless the required pretrained model is already available locally.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO = Path('/kaggle/input/mailsense')
WORK = Path('/kaggle/working/mailsense')

if not REPO.exists():
    raise FileNotFoundError(
        f'MailSense Kaggle dataset was not found at {REPO}. '
        'Attach the dataset containing src/, configs/, data/, and embeddings/.'
    )

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

for name in ('src', 'configs', 'data', 'embeddings'):
    source = REPO / name
    if not source.exists():
        raise FileNotFoundError(f'Missing {source}')
    shutil.copytree(source, WORK / name)

if not (WORK / 'data' / 'Ask0729-fixed.txt').exists():
    raise FileNotFoundError('data/Ask0729-fixed.txt is missing.')

if not (WORK / 'embeddings' / 'word2vec.bin').exists():
    raise FileNotFoundError(
        'embeddings/word2vec.bin is missing. It is required for the LSTM experiment.'
    )

os_cwd = Path.cwd()
import os
os.chdir(WORK)
sys.path.insert(0, str(WORK))

import torch
print('Working directory:', WORK)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Install dependencies

Kaggle images normally already contain most dependencies. This verifies/installs the versions declared by the project.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements.txt')],
    check=True,
)


## Helper for running the finalized source scripts

In [ ]:
def run_module(module):
    command = [sys.executable, '-m', module]
    print('$', ' '.join(command))
    subprocess.run(command, cwd=WORK, check=True)


## 1. Prepare the common dataset

The preprocessing script creates one finalized stratified train/validation/test split. All three models reuse these exact files.

In [ ]:
run_module('src.preprocessing.prepare_data')


## 2. Model A — TF-IDF + Logistic Regression

In [ ]:
run_module('src.tfidf.train_tfidf')


## 3. Model B — LSTM + pretrained Word2Vec

The LSTM loads `embeddings/word2vec.bin` and initializes its embedding layer from those pretrained vectors. The configured embedding layer is trainable during LSTM training.

In [ ]:
run_module('src.lstm.train_lstm')


## 4. Model C — pretrained BERT

The finalized experiment uses standard `bert-base-uncased` full fine-tuning. No encoder layers are frozen and no gradient-accumulation option is used.

In [ ]:
run_module('src.bert.train_bert')


## 5. Compare all three models

In [ ]:
run_module('src.evaluation.compare')
print((WORK / 'results' / 'comparison_test.md').read_text(encoding='utf-8'))


## 6. Inspect final comparison

In [ ]:
import pandas as pd

comparison = WORK / 'results' / 'comparison_test.csv'
if not comparison.exists():
    raise FileNotFoundError(comparison)
display(pd.read_csv(comparison))


## 7. Final artifacts

The final results are under `/kaggle/working/mailsense/results` and model checkpoints under `/kaggle/working/mailsense/models`. Download/copy the result files after the run. Model checkpoints are not intended for Git.

In [ ]:
for path in sorted((WORK / 'results').rglob('*')):
    if path.is_file():
        print(f'{path.stat().st_size:>10,}  {path.relative_to(WORK)}')
